# Boosting Algorithms: AdaBoost and XGBoost

Welcome to the world of **Boosting**! 
Unlike Bagging (Random Forests) where we build multiple independent trees and average them, Boosting builds trees **sequentially**. Each new tree tries to fix the errors made by all the previous trees combined!

In this notebook, we will implement AdaBoost and Gradient Boosting from scratch, and also explore the powerful XGBoost library.


## 1. AdaBoost (Adaptive Boosting)

AdaBoost focuses on the **samples** that previous trees misclassified. 
It does this by giving higher weights to misclassified points so the next tree pays more attention to them.

Let's implement `MyAdaBoost` from scratch!


In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class MyAdaBoost:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.models = []
        self.alphas = []
        
    def fit(self, X, y):
        n_samples = X.shape[0]
        # Initialize weights to 1/N
        w = np.ones(n_samples) / n_samples
        
        for _ in range(self.n_estimators):
            # Create a weak learner (Decision Stump)
            stump = DecisionTreeClassifier(max_depth=1)
            # Fit the weak learner with the current sample weights
            stump.fit(X, y, sample_weight=w)
            
            # Predict
            predictions = stump.predict(X)
            
            # Calculate error
            misclassified = (predictions != y)
            error = np.sum(w[misclassified]) / np.sum(w)
            
            # If error is 0, we can stop
            if error == 0:
                self.models.append(stump)
                self.alphas.append(1.0)
                break
                
            # Calculate alpha (amount of say for this stump)
            # Add a small epsilon to prevent division by zero
            epsilon = 1e-10
            alpha = 0.5 * np.log((1.0 - error + epsilon) / (error + epsilon))
            
            # Update weights
            w = w * np.exp(-alpha * y * predictions)
            
            # Normalize weights
            w = w / np.sum(w)
            
            # Save the stump and its alpha
            self.models.append(stump)
            self.alphas.append(alpha)
            
    def predict(self, X):
        # Final prediction is the sign of the weighted sum of weak learner predictions
        stump_preds = np.array([model.predict(X) for model in self.models])
        weighted_preds = np.dot(self.alphas, stump_preds)
        return np.sign(weighted_preds)

# Let's test MyAdaBoost!
# Note: AdaBoost traditionally works with labels -1 and 1
X, y = make_classification(n_samples=500, n_features=2, n_informative=2, n_redundant=0, random_state=42)
y = np.where(y == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

my_ada = MyAdaBoost(n_estimators=50)
my_ada.fit(X_train, y_train)
y_pred = my_ada.predict(X_test)

print(f"MyAdaBoost Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")


MyAdaBoost Accuracy: 85.00%


## 2. Gradient Boosting and XGBoost

While AdaBoost modifies the *weights of data points*, **Gradient Boosting** instead tries to predict the **residual errors** (differences between the actual values and our predictions) of the previous models!

**XGBoost** (Extreme Gradient Boosting) is simply an extremely optimized, high-performance implementation of Gradient Boosting that includes regularization to prevent overfitting.

Let's implement a simplified `MyGradientBoosting` for Regression, and then see `xgboost` in action!


In [2]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression

class MyGradientBoosting:
    def __init__(self, n_estimators=100, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.models = []
        self.initial_prediction = None
        
    def fit(self, X, y):
        # Start with the mean of y as the initial prediction
        self.initial_prediction = np.mean(y)
        predictions = np.full(len(y), self.initial_prediction)
        
        for _ in range(self.n_estimators):
            # Calculate residuals (actual - predicted)
            residuals = y - predictions
            
            # Fit a shallow decision tree to the residuals
            tree = DecisionTreeRegressor(max_depth=3)
            tree.fit(X, residuals)
            
            # Add the tree's predictions (scaled by learning rate) to our running predictions
            predictions += self.learning_rate * tree.predict(X)
            
            # Save the tree
            self.models.append(tree)
            
    def predict(self, X):
        predictions = np.full(X.shape[0], self.initial_prediction)
        for tree in self.models:
            predictions += self.learning_rate * tree.predict(X)
        return predictions

# Test MyGradientBoosting
X_reg, y_reg = make_regression(n_samples=500, n_features=2, noise=15, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

my_gb = MyGradientBoosting(n_estimators=100, learning_rate=0.1)
my_gb.fit(X_train_r, y_train_r)
y_pred_r = my_gb.predict(X_test_r)

from sklearn.metrics import mean_squared_error
print(f"MyGradientBoosting MSE: {mean_squared_error(y_test_r, y_pred_r):.2f}")


MyGradientBoosting MSE: 230.74


## 3. The Power of XGBoost

For real-world projects, building trees from scratch is slow. This is where `xgboost` comes in! It is lightning fast, supports parallel processing, handles missing values automatically, and uses L1/L2 regularization to combat overfitting.


In [4]:
# If xgboost is not installed, you can install it via: !pip install xgboost
try:
    import xgboost as xgb
    
    # We will use the classification dataset from the AdaBoost section
    # XGBoost expects labels as 0 and 1, so we'll convert back
    y_xgb_train = np.where(y_train == -1, 0, 1)
    y_xgb_test = np.where(y_test == -1, 0, 1)

    xgb_clf = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    
    xgb_clf.fit(X_train, y_xgb_train)
    xgb_preds = xgb_clf.predict(X_test)
    
    print(f"XGBoost Classifier Accuracy: {accuracy_score(y_xgb_test, xgb_preds)*100:.2f}%")
except ImportError:
    print("xgboost is not installed! Run '!pip install xgboost' to use it.")


xgboost is not installed! Run '!pip install xgboost' to use it.


In [5]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 1.2 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 MB 1.0 MB/s eta 0:00:0000:0100:04
